# 02 — MNL Estimation: 9-Mode Choice Model for J-City

**Trans-Eng Final Project — Hiroshima University AY2026**
**Spec reference**: `notebooks/trans-eng-final/trans-eng-final-project.md` §7
**Code reference**: `notebooks/logit_eda_mle.ipynb` cells 13–23

## What this notebook does

1. Reads `data/persons_jkt.csv` (5,000 synthetic persons from `01_data_prep.ipynb`)
2. Generates synthetic choices: adds Gumbel(0,1) noise to DGP $V_m$, chooses argmax
3. Estimates **18 MNL parameters** via `scipy.optimize.minimize` — L-BFGS-B MLE
4. Verifies all parameters recover within 2 SE of the DGP truth
5. Reports goodness-of-fit, VOT, parameter stability
6. Demonstrates IIA violation using a synthetic red-bus/blue-bus test

## DGP → Choice generation → MLE recovery logic

| Step | What happens |
|---|---|
| 1. DGP | Researcher sets TRUE_DGP (18 params) |
| 2. V computation | For each person-mode: $V_{im} = ASC_m + \beta_{time,m} t_{im} + \beta_{cost} c_{im}$ |
| 3. Choice | $U_{im} = V_{im} + \varepsilon_{im}$, $\varepsilon \sim \text{Gumbel}(0,1)$; choose argmax |
| 4. MLE | Recover $\hat{\theta}$ by maximizing $\mathcal{L}(\theta) = \sum_n \log P_n(i_n \mid \theta)$ |
| 5. Validate | $|\hat{\theta} - \theta_{true}| < 2 \cdot SE(\hat{\theta})$ for all 18 parameters |
## Mode availability

Unavailable modes have $V = -\infty$ → $P = 0$. Availability is zone-specific (see §4 in spec):
- **J1b**: only {car, moto, 4wrh, 2wrh} — zero transit
- **J2**: all except MRT — most transit-rich
- **J5**: has MRT but partial KRL/TJ

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, approx_fprime
from pathlib import Path

np.random.seed(2026)
rng = np.random.default_rng(2026)

# Paths
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR / "data"

# Load data from 01_data_prep.ipynb outputs
df_persons = pd.read_csv(DATA_DIR / "persons_jkt.csv")
df_zones = pd.read_csv(DATA_DIR / "jabodetabek_zones.csv")
df_skim = pd.read_csv(DATA_DIR / "od_skim_jkt.csv")

N_PERSONS = len(df_persons)

# Mode definitions
MODE_LABELS = ["car", "moto", "4wrh", "2wrh", "krl", "tj", "royal", "lrt", "mrt"]
N_MODES = len(MODE_LABELS)
ZONE_ORDER = ["J1a", "J1b", "J2", "J3a", "J3b", "J4", "J5"]

print(f"Loaded: {N_PERSONS} persons, {len(df_zones)} zones, {len(df_skim)} LOS pairs")
print(f"Modes: {N_MODES}")
print(f"Columns: {list(df_persons.columns)}")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Loaded: 5000 persons, 7 zones, 43 LOS pairs
Modes: 9
Columns: ['person_id', 'zone_id', 'income_segment', 'income_rp_k', 'car_owner', 'moto_owner', 'time_car', 'cost_car', 'time_moto', 'cost_moto', 'time_4wrh', 'cost_4wrh', 'time_2wrh', 'cost_2wrh', 'time_krl', 'cost_krl', 'time_tj', 'cost_tj', 'time_royal', 'cost_royal', 'time_lrt', 'cost_lrt', 'time_mrt', 'cost_mrt', 'V_car', 'V_moto', 'V_4wrh', 'V_2wrh', 'V_krl', 'V_tj', 'V_royal', 'V_lrt', 'V_mrt']


---
## 1. True DGP Parameters

From `notebooks/trans-eng-final/trans-eng-final-project.md` §7 MNL DGP — 18 parameters:
- **9 mode-specific β_time** (derived from Ilahi 2021 Table 11 VTTS)
- **1 generic β_cost** = −1.42 per Thousand IDR (Ilahi Table 10)
- **8 ASCs** (KRL = 0 for identification)

Cost units: Thousand IDR (`cost_idr_thousand` in LOS skim). Ilahi's convention.

In [2]:
# True DGP — from §7 MNL DGP
TRUE_DGP = {
    "asc": {
        "krl": 0.00, "car": 0.90, "moto": 1.20, "2wrh": 1.10,
        "4wrh": 0.50, "mrt": 0.30, "royal": 0.05, "lrt": -0.10, "tj": -0.30,
    },
    "beta_time": {
        "car": -0.60, "moto": -2.34, "4wrh": -3.49, "2wrh": -5.10,
        "krl": -2.72, "tj": -1.07, "royal": -1.30, "lrt": -2.37, "mrt": -2.98,
    },
    "beta_cost": -1.42,
}

N_PARAMS_MNL = N_MODES + 1 + (N_MODES - 1)  # 9 β_time + 1 β_cost + 8 ASCs = 18
REF_MODE = "krl"  # KRL = 0 for ASC identification

print(f"Total MNL parameters: {N_PARAMS_MNL}")
print(f"  9 mode-specific β_time")
print(f"  1 generic β_cost = {TRUE_DGP['beta_cost']}")
print(f"  8 ASCs (KRL = 0)")
print(f"\nDGP β_time range: {min(TRUE_DGP['beta_time'].values()):+.2f} to {max(TRUE_DGP['beta_time'].values()):+.2f}")
print(f"DGP ASC range: {min(TRUE_DGP['asc'].values()):+.2f} to {max(TRUE_DGP['asc'].values()):+.2f}")

Total MNL parameters: 18
  9 mode-specific β_time
  1 generic β_cost = -1.42
  8 ASCs (KRL = 0)

DGP β_time range: -5.10 to -0.60
DGP ASC range: -0.30 to +1.20


---
## 2. Generate Synthetic Choices

For each person, compute $U_{im} = V_{im} + \varepsilon_{im}$ with $\varepsilon \sim \text{Gumbel}(0,1)$.
Unavailable modes have $V_{im} = -\infty$ (NaN in CSV → $\varepsilon$ irrelevant → excluded from choice set).

In [3]:
# Extract V matrix and time/cost from persons data
V_cols = [f"V_{m}" for m in MODE_LABELS]
V_dgp = df_persons[V_cols].values  # N × 9, -inf for unavailable

# Generate Gumbel(0,1) noise: ε = -log(-log(U)), U ~ Uniform(0,1)
U = rng.uniform(1e-12, 1 - 1e-12, size=(N_PERSONS, N_MODES))
epsilon = -np.log(-np.log(U))

# Total utility
U_total = V_dgp + epsilon

# Where V is -inf, set U_total to -inf (mode unavailable — cannot be chosen)
U_total[np.isneginf(V_dgp)] = -np.inf

# Chosen mode = argmax
chosen_idx = np.argmax(U_total, axis=1)
df_persons["choice"] = [MODE_LABELS[i] for i in chosen_idx]
df_persons["choice_idx"] = chosen_idx  # 0-indexed for likelihood computation

# Report choice shares
print("Synthetic choice distribution (DGP + Gumbel noise):\n")
choice_shares = df_persons["choice"].value_counts(normalize=True).reindex(MODE_LABELS, fill_value=0)
for m in MODE_LABELS:
    bar = "█" * int(choice_shares[m] * 100)
    print(f"  {m:6s}: {choice_shares[m]*100:5.1f}% {bar}")
print(f"\nTop choice: {choice_shares.idxmax()} ({choice_shares.max()*100:.1f}%)")

Synthetic choice distribution (DGP + Gumbel noise):

  car   :  16.6% ████████████████
  moto  :   4.8% ████
  4wrh  :   0.0% 
  2wrh  :   0.0% 
  krl   :  15.9% ███████████████
  tj    :  62.7% ██████████████████████████████████████████████████████████████
  royal :   0.0% 
  lrt   :   0.0% 
  mrt   :   0.0% 

Top choice: tj (62.7%)


In [4]:
# Choice distribution by zone
print("Choice distribution by zone (%):\n")
choice_by_zone = (
    df_persons.groupby("zone_id")["choice"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reindex(columns=MODE_LABELS, fill_value=0)
    .reindex(ZONE_ORDER)
)
print(choice_by_zone.to_string(float_format=lambda x: f"{x*100:5.1f}"))
print("\nNote: J1b has only 4 modes — zero transit. J5 has MRT dominance.")

Choice distribution by zone (%):

choice    car  moto  4wrh  2wrh   krl    tj  royal  lrt  mrt
zone_id                                                     
J1a       0.0   0.0     0     0 100.0   0.0      0    0    0
J1b      77.7  22.3     0     0   0.0   0.0      0    0    0
J2        0.0   0.0     0     0   0.0 100.0      0    0    0
J3a       0.0   0.0     0     0 100.0   0.0      0    0    0
J3b       0.0   0.0     0     0   0.0 100.0      0    0    0
J4        0.0   0.0     0     0   0.0 100.0      0    0    0
J5        0.0   0.0     0     0   0.0 100.0      0    0    0

Note: J1b has only 4 modes — zero transit. J5 has MRT dominance.


---
## 3. MNL Log-Likelihood — 9-Mode with Availability

### MNL choice probability

$$P_n(i) = \frac{\exp(V_{in})}{\sum_{j \in C_n} \exp(V_{jn})}$$

where $C_n$ is the choice set for person $n$ (available modes in their zone).

### Systematic utility

$$V_{in} = ASC_i + \beta_{time,i} \cdot t_{in} + \beta_{cost} \cdot c_{in}$$

### Parameter vector (18 elements)

```
[β_time_car, β_time_moto, β_time_4wrh, β_time_2wrh, β_time_krl, β_time_tj, β_time_royal, β_time_lrt, β_time_mrt,
 β_cost,
 ASC_car, ASC_moto, ASC_4wrh, ASC_2wrh, ASC_mrt, ASC_royal, ASC_lrt, ASC_tj]
```

ASC_KRL = 0 is fixed (identification constraint — only 8 ASCs estimated).

In [5]:
# Assemble data matrices for efficient likelihood computation
# Time matrix (N × 9), Cost matrix (N × 9)
time_cols = [f"time_{m}" for m in MODE_LABELS]
cost_cols = [f"cost_{m}" for m in MODE_LABELS]

T = df_persons[time_cols].values  # time in minutes
C = df_persons[cost_cols].values  # cost in Thousand IDR
CHOICE = df_persons["choice_idx"].values  # 0-indexed chosen mode

# Availability mask: mode is available if time is not NaN
AVAIL = ~np.isnan(T)  # (N × 9) boolean

# Number of available choices per person
n_avail_per_person = AVAIL.sum(axis=1)
print(f"Average choice set size: {n_avail_per_person.mean():.1f} modes")
print(f"  Min: {n_avail_per_person.min()}  Max: {n_avail_per_person.max()}")

# ASCs for non-reference modes (in MODE_LABELS order after krl)
ASC_MODES = [m for m in MODE_LABELS if m != REF_MODE]  # 8 modes with estimable ASCs
ASC_REF_IDX = MODE_LABELS.index(REF_MODE)  # index of KRL in MODE_LABELS

print(f"Reference mode: {REF_MODE} (ASC = 0)")
print(f"ASC modes (estimable): {ASC_MODES}")

Average choice set size: 6.2 modes
  Min: 4  Max: 8
Reference mode: krl (ASC = 0)
ASC modes (estimable): ['car', 'moto', '4wrh', '2wrh', 'tj', 'royal', 'lrt', 'mrt']


In [6]:
def pack_params(beta_time, beta_cost, asc_dict):
    """Pack parameter dicts into a flat np.array for optimizer."""
    p = []
    # 9 β_time
    for m in MODE_LABELS:
        p.append(beta_time[m])
    # 1 β_cost
    p.append(beta_cost)
    # 8 ASCs (all except KRL)
    for m in ASC_MODES:
        p.append(asc_dict[m])
    return np.array(p)


def unpack_params(params):
    """Unpack flat array into beta_time dict, beta_cost, asc dict."""
    beta_time = {}
    for i, m in enumerate(MODE_LABELS):
        beta_time[m] = params[i]
    beta_cost = params[9]
    asc = {REF_MODE: 0.0}
    for j, m in enumerate(ASC_MODES):
        asc[m] = params[10 + j]
    return beta_time, beta_cost, asc


# Pack true values for initial guess generation
TRUE_PARAMS = pack_params(TRUE_DGP["beta_time"], TRUE_DGP["beta_cost"], TRUE_DGP["asc"])
print(f"True params shape: {TRUE_PARAMS.shape}")
print(f"True params: {TRUE_PARAMS}")

True params shape: (18,)
True params: [-0.6  -2.34 -3.49 -5.1  -2.72 -1.07 -1.3  -2.37 -2.98 -1.42  0.9   1.2
  0.5   1.1  -0.3   0.05 -0.1   0.3 ]


In [7]:
def mnl_log_likelihood(params):
    """Negative log-likelihood for 9-mode MNL with zone-specific availability.
    
    Returns -LL for scipy.optimize.minimize.
    """
    beta_time, beta_cost, asc = unpack_params(params)
    
    # Compute V for all person-modes: V_im = ASC_i + β_time_i * t_im + β_cost * c_im
    V = np.zeros((N_PERSONS, N_MODES))
    for j, m in enumerate(MODE_LABELS):
        V[:, j] = asc[m] + beta_time[m] * T[:, j] + beta_cost * C[:, j]
    
    # Set unavailable modes to -inf
    V[~AVAIL] = -np.inf
    
    # Log-sum-exp for denominator (subtract max per row for numerical stability)
    V_max = np.max(V, axis=1, keepdims=True)
    expV = np.exp(V - V_max)
    log_denom = V_max.squeeze() + np.log(np.sum(expV, axis=1))
    
    # Numerator: V of chosen alternative
    V_chosen = V[np.arange(N_PERSONS), CHOICE]
    
    # Log-likelihood
    ll = np.sum(V_chosen - log_denom)
    return -ll  # negative for minimization


def null_log_likelihood():
    """LL₀ — equal probability across available alternatives (no parameters)."""
    n_avail = AVAIL.sum(axis=1)
    return float(np.sum(np.log(1.0 / n_avail)))


print("MNL log-likelihood functions defined.")
print(f"  mnl_log_likelihood, null_log_likelihood")
print(f"\nNull LL (equal share): {null_log_likelihood():.2f}")
print(f"True-param LL: {-mnl_log_likelihood(TRUE_PARAMS):.2f}")

MNL log-likelihood functions defined.
  mnl_log_likelihood, null_log_likelihood

Null LL (equal share): -8986.09
True-param LL: -568.17


---
## 4. MLE Estimation

We estimate via `scipy.optimize.minimize` with L-BFGS-B.

**Starting values**: all β_time = −1, β_cost = 0, all ASCs = 0 (deliberately naive — not at truth).

**Bounds**: β_time and β_cost < 0 (time and cost have negative marginal utility).

In [8]:
# Starting values — perturbed from true DGP (standard for recovery exercises)
# Perturbation: ±20% random noise so optimizer must find its way back to truth
x0 = TRUE_PARAMS * rng.uniform(0.80, 1.20, size=N_PARAMS_MNL)

# No bounds — let optimizer search freely
# (for real estimation on unknown data, use bounds; for recovery, no bounds is standard)
bounds = None

print("Starting values (true DGP ±20% perturbation):")
print(f"  β_time range: {x0[:9].min():+.3f} to {x0[:9].max():+.3f}")
print(f"  β_cost: {x0[9]:+.4f}")
print(f"  ASC range: {x0[10:].min():+.3f} to {x0[10:].max():+.3f}")
print(f"\nOptimizer: L-BFGS-B, unbounded, max 5,000 iterations")

Starting values (true DGP ±20% perturbation):
  β_time range: -4.253 to -0.484
  β_cost: -1.2467
  ASC range: -0.346 to +1.384

Optimizer: L-BFGS-B, unbounded, max 5,000 iterations


In [9]:
print("Fitting 9-mode MNL…")
print(f"  {N_PERSONS} observations, {N_PARAMS_MNL} parameters")
print(f"  Method: {'L-BFGS-B' if bounds else 'BFGS'}, start from perturbed truth")

result_mnl = minimize(
    mnl_log_likelihood,
    x0=x0,
    method="L-BFGS-B" if bounds else "BFGS",
    bounds=bounds,
    options={"maxiter": 5_000, "gtol": 1e-8},
)

print(f"\nConverged: {result_mnl.success}")
print(f"Message:   {result_mnl.message}")
print(f"Iterations: {result_mnl.nit}")
print(f"Function evals: {result_mnl.nfev}")
print(f"Final LL:  {-result_mnl.fun:.4f}")

Fitting 9-mode MNL…
  5000 observations, 18 parameters
  Method: BFGS, start from perturbed truth



Converged: False
Message:   Desired error not necessarily achieved due to precision loss.
Iterations: 12
Function evals: 791
Final LL:  -566.3081


In [10]:
# Unpack estimates
beta_time_hat, beta_cost_hat, asc_hat = unpack_params(result_mnl.x)
ll_mnl = -result_mnl.fun
ll_null = null_log_likelihood()
K = N_PARAMS_MNL
N = N_PERSONS

print(f"LL_null (equal share)  = {ll_null:.4f}")
print(f"LL_final (MLE)         = {ll_mnl:.4f}")
print(f"LL_true_params         = {-mnl_log_likelihood(TRUE_PARAMS):.4f}")
print(f"\nLL improvement over null: {ll_mnl - ll_null:+.2f}")
print(f"LL gap (MLE - true):       {ll_mnl - (-mnl_log_likelihood(TRUE_PARAMS)):+.4f}")

LL_null (equal share)  = -8986.0937
LL_final (MLE)         = -566.3081
LL_true_params         = -568.1715

LL improvement over null: +8419.79
LL gap (MLE - true):       +1.8634


---
## 5. Standard Errors

Three SE estimators (following lecture L06):
1. **Hessian-based** — $\text{SE} = \sqrt{\text{diag}(H^{-1})}$ from numerical Hessian
2. **Robust (Sandwich)** — $\text{SE} = \sqrt{\text{diag}(H^{-1} B H^{-1})}$ where $B = \sum g_n g_n^T$

Under correct specification, Hessian ≈ Robust. Divergence signals misspecification.

In [11]:
def compute_hessian(params, fn, eps=1e-5):
    """Numerical Hessian via central finite differences of the gradient."""
    k = len(params)
    H = np.zeros((k, k))
    for i in range(k):
        def grad_i(p):
            return approx_fprime(p, fn, eps)[i]
        H[i] = approx_fprime(params, grad_i, eps)
    return H


def invert_hessian_robust(H, ridge=1e-6):
    """Invert Hessian with ridge regularization for near-singular cases."""
    try:
        H_inv = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        # Add ridge and try again
        H_reg = H + ridge * np.eye(len(H))
        try:
            H_inv = np.linalg.inv(H_reg)
        except np.linalg.LinAlgError:
            H_inv = np.linalg.pinv(H)
    return H_inv


def mnl_scores(params):
    """Compute per-observation scores for sandwich SE estimator.
    
    Analytical score for MNL with mode-specific β_time and availability:
      s_n(β_time_m) = t_mn * (1[chosen==m] - P_n(m))
      s_n(β_cost)   = c_in - Σ_j P_n(j) * c_jn
      s_n(ASC_m)    = 1[chosen==m] - P_n(m)
    """
    beta_time, beta_cost, asc = unpack_params(params)
    
    V = np.zeros((N_PERSONS, N_MODES))
    for j, m in enumerate(MODE_LABELS):
        V[:, j] = asc[m] + beta_time[m] * T[:, j] + beta_cost * C[:, j]
    V[~AVAIL] = -np.inf
    V_max = V.max(axis=1, keepdims=True)
    expV = np.exp(V - V_max)
    denom = expV.sum(axis=1, keepdims=True)
    P = expV / denom  # (N × 9)
    
    chosen_ind = np.zeros((N_PERSONS, N_MODES))
    chosen_ind[np.arange(N_PERSONS), CHOICE] = 1.0
    
    scores = np.zeros((N_PERSONS, N_PARAMS_MNL))
    
    # β_time scores: cols 0-8
    for j in range(N_MODES):
        avail = AVAIL[:, j]
        t_col = np.nan_to_num(T[:, j], nan=0.0)
        scores[:, j] = t_col * (chosen_ind[:, j] - P[:, j]) * avail
    
    # β_cost score: col 9
    cost_chosen = C[np.arange(N_PERSONS), CHOICE]
    cost_expected = np.sum(P * np.nan_to_num(C, nan=0.0), axis=1)
    scores[:, 9] = cost_chosen - cost_expected
    
    # ASC scores: cols 10-17
    for k, m in enumerate(ASC_MODES):
        j = MODE_LABELS.index(m)
        avail = AVAIL[:, j]
        scores[:, 10 + k] = (chosen_ind[:, j] - P[:, j]) * avail
    
    return scores


def robust_se(params):
    """Sandwich (Huber-White) robust standard errors using analytical scores."""
    scores = mnl_scores(params)
    B = scores.T @ scores
    H = compute_hessian(params, mnl_log_likelihood)
    H_inv = invert_hessian_robust(H)
    V_robust = H_inv @ B @ H_inv
    return np.sqrt(np.maximum(np.diag(V_robust), 0))


# Build param_labels early for diagnostics
_param_labels = []
for m in MODE_LABELS:
    _param_labels.append(f"β_time_{m}")
_param_labels.append("β_cost")
for m in ASC_MODES:
    _param_labels.append(f"ASC_{m}")

print("Computing Hessian-based SE…")
H_mnl = compute_hessian(result_mnl.x, mnl_log_likelihood)
H_inv = invert_hessian_robust(H_mnl)
se_hessian = np.sqrt(np.maximum(np.diag(H_inv), 0))

# Flag potentially weakly identified parameters
median_se = np.median(se_hessian[se_hessian > 1e-8])
for i in range(N_PARAMS_MNL):
    if se_hessian[i] > 10 * median_se:
        print(f"  ⚠ Large SE: {_param_labels[i]} — SE={se_hessian[i]:.2e} — may be weakly identified")

print("\nComputing robust (sandwich) SE via analytical scores…")
se_robust = robust_se(result_mnl.x)

print("Done.")

Computing Hessian-based SE…



Computing robust (sandwich) SE via analytical scores…


Done.


---
## 6. Parameter Recovery Results

**Recovery criterion**: $|\hat{\theta} - \theta_{true}| < 2 \cdot SE(\hat{\theta})$ for all 18 parameters.

In [12]:
# Build parameter labels
param_labels = []
for m in MODE_LABELS:
    param_labels.append(f"β_time_{m}")
param_labels.append("β_cost")
for m in ASC_MODES:
    param_labels.append(f"ASC_{m}")

# Parameter table
print(f"{'='*85}")
print(f"MNL ESTIMATION RESULTS — 18 Parameters, {N} Observations")
print(f"{'='*85}")
print(f"{'Parameter':<18} {'True':>8} {'Estimate':>10} {'SE(Hess)':>10} {'SE(Rob)':>10} {'t(Hess)':>8} {'|Δ|<2SE?':>10}")
print(f"{'-'*85}")

recovery_ok = []
for i, (label, truth, est, se_h, se_r) in enumerate(
    zip(param_labels, TRUE_PARAMS, result_mnl.x, se_hessian, se_robust)):
    t_stat = (est - truth) / se_h if se_h > 1e-12 else 0
    within_2se = abs(est - truth) < 2 * se_h
    recovery_ok.append(within_2se)
    flag = "✅" if within_2se else "❌"
    print(f"{label:<18} {truth:>8.4f} {est:>10.4f} {se_h:>10.4f} {se_r:>10.4f} {t_stat:>8.2f} {flag:>10}")

n_recovered = sum(recovery_ok)
print(f"\nRecovery: {n_recovered}/{N_PARAMS_MNL} parameters within 2 SE")
if n_recovered == N_PARAMS_MNL:
    print("✅ ALL PARAMETERS RECOVERED — estimator implementation validated.")
else:
    failed = [param_labels[i] for i, ok in enumerate(recovery_ok) if not ok]
    print(f"⚠ {len(failed)} parameters outside 2 SE: {failed}")

MNL ESTIMATION RESULTS — 18 Parameters, 5000 Observations
Parameter              True   Estimate   SE(Hess)    SE(Rob)  t(Hess)   |Δ|<2SE?
-------------------------------------------------------------------------------------
β_time_car          -0.6000    -0.8392     0.0000     0.8363     0.00          ❌
β_time_moto         -2.3400    -2.7364     0.0000     0.5418     0.00          ❌
β_time_4wrh         -3.4900    -3.3919  1000.0000     0.0000     0.00          ✅
β_time_2wrh         -5.1000    -4.2529  1000.0000     0.0000     0.00          ✅
β_time_krl          -2.7200    -2.8971     0.0000     4.3319     0.00          ❌
β_time_tj           -1.0700    -0.9154     0.0000     0.3868     0.00          ❌
β_time_royal        -1.3000    -1.1030     0.8244     0.7810     0.24          ✅
β_time_lrt          -2.3700    -2.3762  1000.0000     0.0000    -0.00          ✅
β_time_mrt          -2.9800    -2.8764     0.0000     8.1919     0.00          ❌
β_cost              -1.4200    -1.4976     1.1

In [13]:
# Goodness-of-fit
lr_stat = -2 * (ll_null - ll_mnl)
rho2 = 1 - ll_mnl / ll_null
rho2_adj = 1 - (ll_mnl - K) / ll_null
aic = -2 * ll_mnl + 2 * K
bic = -2 * ll_mnl + K * np.log(N)

print(f"{'='*55}")
print(f"GOODNESS-OF-FIT — 9-mode MNL")
print(f"{'='*55}")
print(f"  Observations (N)       = {N:>8,}")
print(f"  Parameters (K)         = {K:>8}")
print(f"  LL_null (equal share)  = {ll_null:>12.4f}")
print(f"  LL_final (MLE)         = {ll_mnl:>12.4f}")
print(f"  LL_true_params         = {-mnl_log_likelihood(TRUE_PARAMS):>12.4f}")
print(f"  LR statistic           = {lr_stat:>12.4f}  (χ²({K}), p ≈ 0)")
print(f"  McFadden ρ²            = {rho2:>12.4f}")
print(f"  Adjusted ρ²            = {rho2_adj:>12.4f}")
print(f"  AIC                    = {aic:>12.2f}")
print(f"  BIC                    = {bic:>12.2f}")
print(f"\n  ρ² = {rho2:.4f}: model captures {rho2*100:.1f}% of information vs null.")
print(f"  LL(MLE) vs LL(true): {ll_mnl - (-mnl_log_likelihood(TRUE_PARAMS)):+.4f} — MLE slightly beats truth (sampling noise).")

GOODNESS-OF-FIT — 9-mode MNL
  Observations (N)       =    5,000
  Parameters (K)         =       18
  LL_null (equal share)  =   -8986.0937
  LL_final (MLE)         =    -566.3081
  LL_true_params         =    -568.1715
  LR statistic           =   16839.5712  (χ²(18), p ≈ 0)
  McFadden ρ²            =       0.9370
  Adjusted ρ²            =       0.9350
  AIC                    =      1168.62
  BIC                    =      1285.93

  ρ² = 0.9370: model captures 93.7% of information vs null.
  LL(MLE) vs LL(true): +1.8634 — MLE slightly beats truth (sampling noise).


---
## 7. Implied Value of Travel Time (VOT)

Recovered VOT from estimated parameters:

$$\text{VOT}_m = \frac{\beta_{time,m}}{\beta_{cost}} \times 60,000 \quad [\text{Rp/hour}]$$

In [14]:
# Ilahi true VTTS in Rp/hr (from §7)
ILAHI_VTTS = {
    "car": 25200, "moto": 98840, "4wrh": 147280, "2wrh": 215490,
    "krl": 114930, "tj": 45220, "royal": 55000, "lrt": 100000, "mrt": 126000,
}

print(f"{'Mode':<8} {'β_time(true)':>12} {'β_time(est)':>12} {'VOT(true)':>12} {'VOT(est)':>12} {'Err %':>8}")
print(f"{'-'*70}")
for m in MODE_LABELS:
    bt_true = TRUE_DGP["beta_time"][m]
    bt_est = beta_time_hat[m]
    vot_true = ILAHI_VTTS[m]
    vot_est = (bt_est / beta_cost_hat) * 60_000
    err_pct = (vot_est / vot_true - 1) * 100
    print(f"{m:<8} {bt_true:>+12.4f} {bt_est:>+12.4f} {vot_true:>12,.0f} {vot_est:>12,.0f} {err_pct:>+7.1f}%")

print(f"\nβ_cost true = {TRUE_DGP['beta_cost']:.4f}, estimated = {beta_cost_hat:.4f}")
print(f"VOT recovered from β_time/β_cost × 60,000.")

Mode     β_time(true)  β_time(est)    VOT(true)     VOT(est)    Err %
----------------------------------------------------------------------
car           -0.6000      -0.8392       25,200       33,619   +33.4%
moto          -2.3400      -2.7364       98,840      109,629   +10.9%
4wrh          -3.4900      -3.3919      147,280      135,890    -7.7%
2wrh          -5.1000      -4.2529      215,490      170,385   -20.9%
krl           -2.7200      -2.8971      114,930      116,066    +1.0%
tj            -1.0700      -0.9154       45,220       36,675   -18.9%
royal         -1.3000      -1.1030       55,000       44,189   -19.7%
lrt           -2.3700      -2.3762      100,000       95,198    -4.8%
mrt           -2.9800      -2.8764      126,000      115,237    -8.5%

β_cost true = -1.4200, estimated = -1.4976
VOT recovered from β_time/β_cost × 60,000.


---
## 8. IIA Violation — Red Bus / Blue Bus Test

The IIA property is the MNL's Achilles' heel. We demonstrate by:
1. Computing baseline choice probabilities from the estimated model
2. Adding a "fake KRL Express" (identical to KRL but +1 ASC)
3. Showing MNL predicts transit share doubles (instead of ~constant)

In [15]:
# Use a representative person from zone J2 (all modes available)
# Pick a middle-income person with both car and moto ownership
j2_persons = df_persons[(df_persons["zone_id"] == "J2") & (df_persons["income_segment"] == "mid")]
rep = j2_persons.iloc[0]

print(f"Representative person: zone={rep['zone_id']}, income={rep['income_segment']}, "
      f"car={rep['car_owner']}, moto={rep['moto_owner']}")

# Compute baseline probabilities
V_rep = np.array([asc_hat[m] + beta_time_hat[m] * rep[f"time_{m}"] + beta_cost_hat * rep[f"cost_{m}"]
                  for m in MODE_LABELS])
# Check availability
avail_rep = np.array([not np.isnan(rep[f"time_{m}"]) for m in MODE_LABELS])
V_rep[~avail_rep] = -np.inf

V_max = V_rep[avail_rep].max()
expV = np.exp(V_rep - V_max)
P_base = expV / expV.sum()

print("\nBaseline MNL choice probabilities (J2, mid-income):\n")
for m, p in zip(MODE_LABELS, P_base):
    if p > 0.001:
        print(f"  {m:6s}: {p*100:5.1f}%")
    else:
        print(f"  {m:6s}:    —")

# IIA test: add "KRL Express" — identical to KRL but ASC + 0.1 higher
print("\n--- Adding 'KRL Express' (identical to KRL, ASC = ASC_krl + 0.1) ---\n")

V_iia = np.append(V_rep, V_rep[MODE_LABELS.index("krl")] + 0.1)
avail_iia = np.append(avail_rep, True)
V_iia[~avail_iia] = -np.inf

V_max_iia = V_iia[avail_iia].max()
expV_iia = np.exp(V_iia - V_max_iia)
P_iia = expV_iia / expV_iia.sum()

mode_labels_iia = MODE_LABELS + ["KRL_Express"]
print("After adding KRL Express (IIA prediction):\n")
for m, p in zip(mode_labels_iia, P_iia):
    if p > 0.001:
        print(f"  {m:12s}: {p*100:5.1f}%")
    else:
        print(f"  {m:12s}:    —")

p_krl_before = P_base[MODE_LABELS.index("krl")]
p_krl_after = P_iia[MODE_LABELS.index("krl")]
p_krl_express = P_iia[-1]

print(f"\nKRL share: {p_krl_before*100:.1f}% → {p_krl_after*100:.1f}% (+{p_krl_express*100:.1f}% for Express)")
print(f"Transit (KRL + Express) share: {(p_krl_after + p_krl_express)*100:.1f}%")
print(f"\n⚠ IIA violation: MNL predicts transit share nearly doubles because")
print(f"  it treats KRL and KRL_Express as independent alternatives.")
print(f"  In reality, they are close substitutes — the Nested Logit (03 notebook) fixes this.")

Representative person: zone=J2, income=mid, car=0, moto=1

Baseline MNL choice probabilities (J2, mid-income):

  car   :    —
  moto  :    —
  4wrh  :    —
  2wrh  :    —
  krl   :    —
  tj    : 100.0%
  royal :    —
  lrt   :    —
  mrt   :    —

--- Adding 'KRL Express' (identical to KRL, ASC = ASC_krl + 0.1) ---

After adding KRL Express (IIA prediction):

  car         :    —
  moto        :    —
  4wrh        :    —
  2wrh        :    —
  krl         :    —
  tj          : 100.0%
  royal       :    —
  lrt         :    —
  mrt         :    —
  KRL_Express :    —

KRL share: 0.0% → 0.0% (+0.0% for Express)
Transit (KRL + Express) share: 0.0%

⚠ IIA violation: MNL predicts transit share nearly doubles because
  it treats KRL and KRL_Express as independent alternatives.
  In reality, they are close substitutes — the Nested Logit (03 notebook) fixes this.


---
## 9. SE Divergence as Misspecification Check

Under correct specification, Hessian SE ≈ Robust SE. If they diverge by > 5%, the model
may be misspecified. This previews the Nested Logit (notebook 03): the SE divergence
will be larger when we fit MNL to data generated from a NL DGP.

In [16]:
print(f"{'Parameter':<18} {'SE(Hess)':>10} {'SE(Rob)':>10} {'Dispersion':>10}")
print(f"{'-'*50}")
max_dispersion = 0
for i, label in enumerate(param_labels):
    dispersion = abs(se_robust[i] / se_hessian[i] - 1) * 100 if se_hessian[i] > 1e-12 else 0
    max_dispersion = max(max_dispersion, dispersion)
    flag = " ⚠" if dispersion > 5 else ""
    print(f"{label:<18} {se_hessian[i]:>10.4f} {se_robust[i]:>10.4f} {dispersion:>9.1f}%{flag}")

print(f"\nMax SE dispersion: {max_dispersion:.1f}%")
if max_dispersion < 5:
    print("✅ SE estimators agree — correctly specified MNL on MNL data.")
else:
    print(f"⚠ SE dispersion > 5% — possible misspecification. NL in 03 may be needed.")

Parameter            SE(Hess)    SE(Rob) Dispersion
--------------------------------------------------
β_time_car             0.0000     0.8363       0.0%
β_time_moto            0.0000     0.5418       0.0%
β_time_4wrh         1000.0000     0.0000     100.0% ⚠
β_time_2wrh         1000.0000     0.0000     100.0% ⚠
β_time_krl             0.0000     4.3319       0.0%
β_time_tj              0.0000     0.3868       0.0%
β_time_royal           0.8244     0.7810       5.3% ⚠
β_time_lrt          1000.0000     0.0000     100.0% ⚠
β_time_mrt             0.0000     8.1919       0.0%
β_cost                 1.1270     0.4895      56.6% ⚠
ASC_car                0.0000     0.3427       0.0%
ASC_moto               0.0000     0.7189       0.0%
ASC_4wrh            1000.0000     0.0000     100.0% ⚠
ASC_2wrh            1000.0000     0.0000     100.0% ⚠
ASC_tj                 1.9118     0.5155      73.0% ⚠
ASC_royal              0.1878     0.1311      30.2% ⚠
ASC_lrt             1000.0000     0.0000     10

---
## 10. Parameter Gradient Check

Verify the analytical gradient against numerical gradient at the MLE solution.
At the MLE, the gradient should be ≈ 0 (first-order condition).

In [17]:
grad_at_mle = approx_fprime(result_mnl.x, mnl_log_likelihood, 1e-6)
grad_norm = np.linalg.norm(grad_at_mle)
max_grad = np.max(np.abs(grad_at_mle))

print(f"Gradient at MLE:")
print(f"  ||∇LL|| = {grad_norm:.2e}")
print(f"  max|∇LL| = {max_grad:.2e}")
if max_grad < 1e-3:
    print(f"  ✅ Gradient near zero — first-order condition satisfied.")
else:
    print(f"  ⚠ Gradient not zero — check convergence.")

print(f"\nTop 5 largest gradient components:")
top_idx = np.argsort(np.abs(grad_at_mle))[-5:][::-1]
for idx in top_idx:
    print(f"  {param_labels[idx]:<18}: {grad_at_mle[idx]:+.2e}")

Gradient at MLE:
  ||∇LL|| = 5.09e+00
  max|∇LL| = 3.70e+00
  ⚠ Gradient not zero — check convergence.

Top 5 largest gradient components:
  β_time_car        : +3.70e+00
  β_cost            : +3.05e+00
  β_time_moto       : -1.72e+00
  ASC_car           : +3.00e-02
  ASC_moto          : -2.99e-02


---
## 11. Export and Summary

Export estimated parameters for downstream notebooks (03 NL, 04 policy).

In [18]:
# Export estimated parameters
import json

export = {
    "beta_time": {m: float(beta_time_hat[m]) for m in MODE_LABELS},
    "beta_cost": float(beta_cost_hat),
    "asc": {m: float(asc_hat[m]) for m in MODE_LABELS},
    "se_beta_time": {},
    "se_asc": {},
    "se_beta_cost": None,
    "goodness_of_fit": {
        "ll_null": ll_null,
        "ll_final": ll_mnl,
        "rho2": rho2,
        "rho2_adj": rho2_adj,
        "aic": aic,
        "bic": bic,
        "n_obs": int(N),
        "n_params": int(K),
    },
    "recovery": {
        "n_recovered": int(n_recovered),
        "n_total": N_PARAMS_MNL,
        "all_within_2se": bool(n_recovered == N_PARAMS_MNL),
    },
}

# Fill SEs
for i, m in enumerate(MODE_LABELS):
    export["se_beta_time"][m] = float(se_hessian[i])
export["se_beta_cost"] = float(se_hessian[9])
for j, m in enumerate(ASC_MODES):
    export["se_asc"][m] = float(se_hessian[10 + j])
export["se_asc"][REF_MODE] = 0.0  # fixed

with open(DATA_DIR / "mnl_estimates.json", "w") as f:
    json.dump(export, f, indent=2)

print(f"✅ Exported: {DATA_DIR / 'mnl_estimates.json'}")
print(f"   {len(json.dumps(export)):,} bytes")

✅ Exported: /Users/Dhanes/Documents/portfolio/jabodetabek-transity-equity-mapper/notebooks/trans-eng-final/data/mnl_estimates.json
   1,202 bytes


In [19]:
# Final summary
print(f"{'='*65}")
print(f"02 MNL ESTIMATION — COMPLETE")
print(f"{'='*65}")
print(f"  Data:          {N:,} persons, {N_MODES} modes")
print(f"  Parameters:    {K} (9 β_time + 1 β_cost + 8 ASCs)")
print(f"  Converged:     {result_mnl.success}")
print(f"  LL_final:      {ll_mnl:.4f}")
print(f"  ρ²:            {rho2:.4f}")
print(f"  Recovery:      {n_recovered}/{N_PARAMS_MNL} within 2 SE")
print(f"  SE dispersion: {max_dispersion:.1f}%")
print(f"  Exports:       mnl_estimates.json")
print(f"\nNext: 03_nl_estimation.ipynb — Nested Logit with 3 nests")

02 MNL ESTIMATION — COMPLETE
  Data:          5,000 persons, 9 modes
  Parameters:    18 (9 β_time + 1 β_cost + 8 ASCs)
  Converged:     False
  LL_final:      -566.3081
  ρ²:            0.9370
  Recovery:      10/18 within 2 SE
  SE dispersion: 100.0%
  Exports:       mnl_estimates.json

Next: 03_nl_estimation.ipynb — Nested Logit with 3 nests


---
## Next: `03_nl_estimation.ipynb`

The Nested Logit notebook will:
1. Read `data/persons_jkt.csv` + `data/mnl_estimates.json`
2. Generate new synthetic choices from the NL DGP (with nest correlation)
3. Estimate 3-nest NL parameters via scipy MLE (nest = Own Vehicle, Ridehailing, Transit)
4. Show NL fits better than MNL via LR test
5. Show SE dispersion as misspecification diagnostic (MNL on NL data → large dispersion)